# Cross-temporal LDA decoding and RSA
Stroud-style dynamic-coding test on single-object trials. Two 3-position sessions: **DMFC = Perle 2022-06-01**, **FEF = Perle 2022-06-04**. Two analyses per region: decode identity (marginal over position) and decode position (marginal over identity), 3 classes, shrinkage-LDA over a pseudopopulation (obs mask), 4-fold CV. A stable/gain code fills the off-diagonal (train-early generalises to test-late); a rotating/dynamic code concentrates accuracy near the diagonal.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / 'pyproject.toml').exists():
    root = root.parent
sys.path.insert(0, str(root / 'scripts'))

import numpy as np
import matplotlib.pyplot as plt
from analysis import decode, rsa, rate_tensor, BINS, subspace_trajectories_group, select_sessions, decode_group, rsa_group, bin_edges, subspace_trajectories, WINDOWS
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SESSIONS = {'DMFC': '2022-06-01', 'FEF': '2022-06-04'}
FACTORS = ['identity', 'position']
centers = BINS[:-1] + 0.05
ext = [0, 2, 0, 2]

In [ ]:
results = {}
for region, sess in SESSIONS.items():
    data = rate_tensor(sess, region, quality="good")          # load once per session/region
    for factor in FACTORS:
        acc, nU = decode(sess, region, factor, data=data)
        sim = rsa(sess, region, factor, data=data)
        results[(region, factor)] = dict(acc=acc, sim=sim, nU=nU)
        print(region, factor, '| units', nU, '| diag', round(np.diag(acc).mean(), 3),
              '| off-diag(train0-1s,test1-2s)', round(acc[:10, 10:].mean(), 3))

## Cross-temporal decoding matrices
Row = region, column = analysis. Dashed lines mark delay onset (1.0 s).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.5))
for r, region in enumerate(SESSIONS):
    for c, factor in enumerate(FACTORS):
        ax = axes[r, c]
        res = results[(region, factor)]
        im = ax.imshow(res['acc'], origin='lower', extent=ext, aspect='equal',
                       cmap='gray_r', vmin=1/3)
        ax.axhline(1.0, color='w', lw=0.6, ls='--'); ax.axvline(1.0, color='w', lw=0.6, ls='--')
        ax.set_title(f"{region} {factor} (n={res['nU']})")
        ax.set_xlabel('test time (s)'); ax.set_ylabel('train time (s)')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='accuracy')
fig.suptitle('Cross-temporal LDA decoding (chance = 1/3)\nData: single-object, 3-position trials, good units, Subject P\nFEF: 2022-06-04; DMFC: 2022-06-01', fontsize=14)
fig.tight_layout()

## Within-time (diagonal) decoding

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for region in SESSIONS:
    for factor in FACTORS:
        ax.plot(centers, np.diag(results[(region, factor)]['acc']), label=f'{region} {factor}')
ax.axhline(1/3, color='k', ls=':', lw=0.8); ax.axvline(1.0, color='k', ls='--', lw=0.6)
ax.set_xlabel('time from stim onset (s)'); ax.set_ylabel('decoding accuracy')
ax.set_title('Within-time decoding'); ax.legend(fontsize=8)
fig.tight_layout()

## Cross-temporal RSA
Cross-validated cosine similarity of class-coding vectors across time. Off-diagonal decay = rotation of the coding subspace.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.5))
for r, region in enumerate(SESSIONS):
    for c, factor in enumerate(FACTORS):
        ax = axes[r, c]
        sim = results[(region, factor)]['sim']
        v = float(np.nanmax(np.abs(sim)))
        im = ax.imshow(sim, origin='lower', extent=ext, aspect='equal',
                       cmap='gray_r', vmin=-v, vmax=v)
        ax.axhline(1.0, color='k', lw=0.5, ls='--'); ax.axvline(1.0, color='k', lw=0.5, ls='--')
        ax.set_title(f'{region} {factor} (n={res['nU']})')
        ax.set_xlabel('time (s)'); ax.set_ylabel('time (s)')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='cosine sim')
fig.suptitle('Cross-temporal RSA (cross-validated cosine of coding vectors)\nData: single-object, 3-position trials, good units, Subject P\nFEF: 2022-06-04; DMFC: 2022-06-01')
fig.tight_layout()

# Group analysis: average / pool over sessions
Select sessions by criteria (per-region MUA/good units, min one-object trials, subject, 3-position design), then aggregate the cross-temporal matrices. `method='average'` runs each session's own pseudopopulation and means the matrices (session = replicate); `method='pool'` concatenates units across sessions into one big pseudopopulation. Position decoding requires 3-position sessions. Set `BIN_SIZE` to change the matrix resolution (0.05 -> 40x40, 0.1 -> 20x20).

In [ ]:
CRITERIA = dict(min_mua=0, min_good=30, min_trials=100, subject='Perle', three_position=True)
METHOD   = 'pool'      # 'average' or 'pool'
BIN_SIZE = 0.05
FACTORS  = ['identity', 'position']
REGIONS  = ['DMFC', 'FEF']
GKW = dict(bin_size=BIN_SIZE, n_jobs=-1)   # n_jobs=-1 uses all cores; add n_repeats=/min_obs= to trade speed/precision

gsess = {reg: select_sessions(region=reg, **CRITERIA) for reg in REGIONS}
for reg in REGIONS:
    print(reg, '->', len(gsess[reg]), 'sessions:', [s for _, s in gsess[reg]])

In [ ]:
gres = {}
for reg in REGIONS:
    for fac in FACTORS:
        d = decode_group(gsess[reg], reg, fac, method=METHOD, **GKW)
        s = rsa_group(gsess[reg], reg, fac, method=METHOD, **GKW)
        gres[(reg, fac)] = dict(acc=d['mean'], sim=s['mean'], meta=d)
        nb_ = d.get('nU', 'pool' if METHOD == 'pool' else '-')
        di = int(round(1.0 / BIN_SIZE))   # delay-onset bin
        print(reg, fac, '| n_sess', d['n_sessions'],
              '| diag', round(float(__import__('numpy').diag(d['mean']).mean()), 3),
              '| off-diag(stim->delay)', round(float(d['mean'][:di, di:].mean()), 3))

## Group cross-temporal decoding matrices

In [ ]:
edges = bin_edges(BIN_SIZE); ext = [0, edges[-1], 0, edges[-1]]
fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.5), dpi=150)
for r, reg in enumerate(REGIONS):
    for c, fac in enumerate(FACTORS):
        ax = axes[r, c]; res = gres[(reg, fac)]
        im = ax.imshow(res['acc'], origin='lower', extent=ext, aspect='equal', cmap='viridis', vmin=1/3)
        ax.axhline(1.0, color='w', lw=0.6, ls='--'); ax.axvline(1.0, color='w', lw=0.6, ls='--')
        ax.set_title(f"{reg} {fac} (n_sess={res['meta']['n_sessions']})")
        ax.set_xlabel('test time (s)'); ax.set_ylabel('train time (s)')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='accuracy')
fig.suptitle(f'Group cross-temporal decoding ({METHOD}, chance=1/3)\nData: single-object, 3-position, ≥30 good units per session, Subject P'); fig.tight_layout()

## Group cross-temporal RSA

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.5), dpi=150)
for r, reg in enumerate(REGIONS):
    for c, fac in enumerate(FACTORS):
        ax = axes[r, c]; sim = gres[(reg, fac)]['sim']
        v = float(np.nanmax(np.abs(sim)))
        im = ax.imshow(sim, origin='lower', extent=ext, aspect='equal', cmap='coolwarm', vmin=-v, vmax=v)
        ax.axhline(1.0, color='k', lw=0.5, ls='--'); ax.axvline(1.0, color='k', lw=0.5, ls='--')
        ax.set_title(f'{reg} {fac}'); ax.set_xlabel('time (s)'); ax.set_ylabel('time (s)')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='cosine sim')
fig.suptitle(f'Group cross-temporal RSA ({METHOD})\nData: single-object, 3-position, ≥30 good units per session, Subject P'); fig.tight_layout()

# Per-window LDA subspaces and projected trajectories (Xie Fig 3D)
Shrinkage LDA is fit on the window-averaged pseudopopulation for each of five windows (200-500, 700-1000, 1000-1300, 1350-1650, 1700-2000 ms). The top-2 right singular vectors of the (class x unit) coefficient matrix, accumulated over folds, give an orthonormal 2D coding plane per window. Held-out class-mean trajectories are projected onto each plane; principal angles between planes quantify subspace rotation. DMFC 2022-06-01, FEF 2022-06-04, good units, Subject P.

In [ ]:
sub = {}
for region, sess in SESSIONS.items():
    for factor in FACTORS:
        res = subspace_trajectories(sess, region, factor)
        sub[(region, factor)] = res
        off = res['angles'][np.triu_indices(len(WINDOWS), 1)]
        print(region, factor, '| units', res['nU'],
              '| mean off-diagonal angle', round(off.mean(), 1), 'deg')

In [ ]:
wlabels = [f'{int(a*1000)}-{int(b*1000)}ms' for a, b in WINDOWS]
palette = ['#1f77b4', '#ff7f0e', '#2ca02c']  # tab10 first three
keys = [(reg, fac) for reg in SESSIONS for fac in FACTORS]
t = BINS[:-1] + (BINS[1] - BINS[0]) / 2  # bin-centre times (s) -> z-axis

for key in keys:
    res = sub[key]
    fig = make_subplots(rows=1, cols=len(WINDOWS), specs=[[{'type': 'scene'}] * len(WINDOWS)],
                        subplot_titles=wlabels, horizontal_spacing=0.01)
    for w in range(len(WINDOWS)):
        P = res['traj'][w]  # (nClass, 2, nbin)
        for ci, c in enumerate(res['classes']):
            xy = P[ci]
            col = palette[ci % len(palette)]
            fig.add_trace(go.Scatter3d(
                x=xy[0], y=xy[1], z=t, mode='lines', line=dict(color=col, width=4),
                name=f'{key[1]} {c}', legendgroup=str(c), showlegend=(w == 0),
                hovertemplate='t=%{z:.2f}s'), row=1, col=w + 1)
            fig.add_trace(go.Scatter3d(  # o = stim onset, diamond = delay end
                x=[xy[0, 0], xy[0, -1]], y=[xy[1, 0], xy[1, -1]], z=[t[0], t[-1]],
                mode='markers', marker=dict(color=col, size=[6, 9],
                                            symbol=['circle-open', 'diamond']),
                legendgroup=str(c), showlegend=False, hoverinfo='skip'), row=1, col=w + 1)
    for i in range(1, len(WINDOWS) + 1):
        fig.update_layout(**{f'scene{i if i > 1 else ""}': dict(
            xaxis_title='', yaxis_title='', zaxis_title='time (s)',
            xaxis=dict(showticklabels=False), yaxis=dict(showticklabels=False))})
    fig.update_layout(height=430, width=1550, margin=dict(l=0, r=0, t=70, b=0),
                      title=f'{key[0]} {key[1]} (n={res["nU"]}) — per-window LDA planes, z = time '
                            '(o = stim onset, diamond = delay end)')
    fig.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.5), dpi=150)
for ax, key in zip(axes.flat, keys):
    A = sub[key]['angles']
    im = ax.imshow(A, cmap='magma', vmin=0, vmax=90)
    ax.set_xticks(range(len(WINDOWS))); ax.set_yticks(range(len(WINDOWS)))
    ax.set_xticklabels(wlabels, rotation=90, fontsize=7); ax.set_yticklabels(wlabels, fontsize=7)
    ax.set_title(f'{key[0]} {key[1]}')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='mean principal angle (deg)')
fig.suptitle('Principal angles between per-window coding subspaces (0 = identical plane, 90 = orthogonal)')
fig.tight_layout()

# Group and conjunction trajectories
Pooled subspaces over the selected sessions (`gsess`, units concatenated into one pseudopopulation). Position/identity give 3 trajectories; conjunction gives 9 (3 objects x 3 positions) projected onto the 3-class **position** plane, so co-located objects sharing a plane location = gain-like, separated = slot-like. `method='average'` instead aggregates only the 5x5 principal-angle matrices (`angles_mean`/`angles_sem`), since per-session planes have arbitrary orientation. Conjunction is 3-position sessions only.

In [ ]:
BASE3 = ['#1f77b4', '#ff7f0e', '#2ca02c']
POS_HUE = {0: ['#9ecae1', '#4292c6', '#08519c'],   # blue,   identity a/b/c light->dark
           1: ['#fdae6b', '#f16913', '#a63603'],   # orange
           2: ['#a1d99b', '#41ab5d', '#006d2c']}   # green
ID_LBL = ['a', 'b', 'c']

def simple_styles(classes, prefix):
    return [dict(color=BASE3[i % 3], name=f'{prefix} {classes[i]}') for i in range(len(classes))]

def conj_styles(classes):
    return [dict(color=POS_HUE[int(c) // 3][int(c) % 3], name=f'pos{int(c) // 3} {ID_LBL[int(c) % 3]}')
            for c in classes]

def plot_traj3d(res, styles, title, height=430, width=1550):
    wl = [f'{int(a*1000)}-{int(b*1000)}ms' for a, b in WINDOWS]
    t = BINS[:-1] + (BINS[1] - BINS[0]) / 2
    fig = make_subplots(rows=1, cols=len(WINDOWS), specs=[[{'type': 'scene'}] * len(WINDOWS)],
                        subplot_titles=wl, horizontal_spacing=0.01)
    for w in range(len(WINDOWS)):
        P = res['traj'][w]  # (nClass, 2, nbin)
        for k in range(P.shape[0]):
            xy, st = P[k], styles[k]
            fig.add_trace(go.Scatter3d(x=xy[0], y=xy[1], z=t, mode='lines',
                line=dict(color=st['color'], width=4), name=st['name'], legendgroup=st['name'],
                showlegend=(w == 0), hovertemplate='t=%{z:.2f}s'), row=1, col=w + 1)
            fig.add_trace(go.Scatter3d(x=[xy[0, 0], xy[0, -1]], y=[xy[1, 0], xy[1, -1]],
                z=[t[0], t[-1]], mode='markers', marker=dict(color=st['color'], size=[6, 9],
                symbol=['circle-open', 'diamond']), legendgroup=st['name'], showlegend=False,
                hoverinfo='skip'), row=1, col=w + 1)
    for i in range(1, len(WINDOWS) + 1):
        fig.update_layout(**{f'scene{i if i > 1 else ""}': dict(xaxis_title='', yaxis_title='',
            zaxis_title='time (s)', xaxis=dict(showticklabels=False), yaxis=dict(showticklabels=False))})
    fig.update_layout(height=height, width=width, margin=dict(l=0, r=0, t=70, b=0), title=title)
    fig.show()

In [ ]:
gsub = {}
for reg in REGIONS:
    for fac in ['position', 'identity']:
        gsub[(reg, fac)] = subspace_trajectories_group(gsess[reg], reg, fac, method='pool')
    gsub[(reg, 'conjunction')] = subspace_trajectories_group(
        gsess[reg], reg, 'conjunction', subspace_factor='position', method='pool')
    print(reg, '| pool nU: position', gsub[(reg, 'position')]['nU'],
          '| conjunction', gsub[(reg, 'conjunction')]['nU'])

In [ ]:
for reg in REGIONS:
    for fac in ['position', 'identity']:
        res = gsub[(reg, fac)]
        plot_traj3d(res, simple_styles(res['classes'], fac),
                    f"{reg} {fac} (pool, n={res['nU']}, n_sess={res['n_sessions']}) — z = time")

In [ ]:
for reg in REGIONS:
    res = gsub[(reg, 'conjunction')]
    plot_traj3d(res, conj_styles(res['classes']),
                f"{reg} conjunction on position plane (pool, n={res['nU']}) — hue = position, shade = identity")